# Assignment 5: Spark Streaming – Order Recovery

## Objective of the Assignment

To use Spark Structured Streaming to process shop orders, remove invalid and duplicate records, and save valid orders in an SQLite database. We will also stop the stream intentionally and restart it using the same checkpoint to check whether any orders are lost or repeated.

The order data is created in this notebook, so no external dataset is needed. Run the cells in order in Google Colab.


In [1]:
%pip -q install pyspark==4.0.1 pandas==2.2.3

^C
Note: you may need to restart the kernel to use updated packages.


## 1. Set up Spark and storage

We create folders for incoming files and Spark checkpoints. SQLite will store our orders. The checkpoint helps Spark continue from where it stopped.


In [ ]:
import json
import os
import sqlite3
import sys
from datetime import datetime, timedelta
from pathlib import Path
from tempfile import mkdtemp

import pandas as pd
from IPython.display import display
from pyspark.errors import StreamingQueryException
from pyspark.sql import SparkSession, functions as F

# Keep all files for this run in one temporary folder.
os.environ["PYSPARK_PYTHON"] = sys.executable
folder = Path(mkdtemp(prefix="spark_orders_"))
incoming = folder / "incoming"
staging = folder / "staging"
checkpoint = folder / "checkpoint"
database = folder / "orders.db"
failure_flag = folder / "stop_once"
stopped_flag = folder / "already_stopped"
incoming.mkdir()
staging.mkdir()

spark = (SparkSession.builder
         .master("local[2]")
         .appName("ShopOrderStreaming")
         .config("spark.sql.shuffle.partitions", "2")
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

# One table stores the orders, and the other remembers completed batches.
with sqlite3.connect(database) as con:
    con.execute("""CREATE TABLE orders (
        order_id TEXT PRIMARY KEY,
        placed_at TEXT,
        department TEXT,
        items INTEGER,
        unit_price_cents INTEGER
    )""")
    con.execute("""CREATE TABLE processed_batches (
        batch_id INTEGER PRIMARY KEY,
        input_rows INTEGER
    )""")


## 2. Create and send sample orders

Our dataset has 18 orders from three shop departments: Groceries, Clothing and Home. We send the data in three deliveries. Each delivery also includes a repeated order and one incorrect record to test the filtering.

Prices are stored in paise (integer values) to avoid rounding errors.


In [ ]:
departments = ["Groceries", "Clothing", "Home"]
prices = [8500, 12000, 6500]  # prices in paise
start_time = datetime(2026, 11, 10, 10, 0)

# Create 18 orders with different IDs, departments and quantities.
orders = []
for i in range(18):
    order = {
        "order_id": f"ORD-{i + 1:03d}",
        "placed_at": (start_time + timedelta(seconds=i * 10)).isoformat(),
        "department": departments[i % 3],
        "items": (i % 3) + 1,
        "unit_price_cents": prices[i % 3]
    }
    orders.append(order)

# Save a delivery as a JSON file in the folder Spark watches.
def send_orders(part, records, bad_line):
    filename = f"orders_{part}.json"
    temporary = staging / filename
    destination = incoming / filename
    lines = [json.dumps(record) for record in records]
    lines.append(bad_line)
    temporary.write_text("\n".join(lines) + "\n", encoding="utf-8")
    temporary.replace(destination)
    print("Delivery", part, "sent:", len(lines), "lines")

# Display data from the SQLite database.
def show_sql(query):
    with sqlite3.connect(database) as con:
        return pd.read_sql_query(query, con)

# First six orders, one duplicate, and one invalid JSON line.
send_orders(1, orders[:6] + [orders[1]], "{wrong json")


## 3. Read and clean the streaming data

Spark reads the JSON files as they arrive. We keep only orders with valid details and remove repeated order IDs using a two-minute watermark.


In [ ]:
schema = """
    order_id STRING,
    placed_at TIMESTAMP,
    department STRING,
    items INT,
    unit_price_cents INT,
    bad_record STRING
"""

stream = (spark.readStream
          .schema(schema)
          .option("mode", "PERMISSIVE")
          .option("columnNameOfCorruptRecord", "bad_record")
          .option("maxFilesPerTrigger", 1)
          .json(str(incoming)))

# Remove incomplete, invalid and incorrect orders.
clean = stream.filter(
    F.col("bad_record").isNull()
    & F.col("order_id").rlike("^ORD-[0-9]{3}$")
    & F.col("placed_at").isNotNull()
    & F.col("department").isin(departments)
    & (F.col("items") > 0)
    & (F.col("unit_price_cents") > 0)
)

unique_orders = (clean
    .withWatermark("placed_at", "2 minutes")
    .dropDuplicatesWithinWatermark(["order_id"])
    .select("order_id", "placed_at", "department", "items", "unit_price_cents"))


## 4. Save each batch to SQLite

We use `foreachBatch` to save the streaming data. Each batch is saved as one database transaction. If Spark reads the same batch after a restart, the saved batch ID prevents us from inserting it again.


In [ ]:
def save_batch(batch, batch_id):
    with sqlite3.connect(database) as con:
        # Skip batches that were already saved before a restart.
        already_saved = con.execute(
            "SELECT 1 FROM processed_batches WHERE batch_id = ?", (int(batch_id),)
        ).fetchone()
        if already_saved:
            batch.count()  # Finish processing Spark's replayed batch.
            return

        formatted = batch.select(
            "order_id",
            F.date_format("placed_at", "yyyy-MM-dd'T'HH:mm:ss").alias("placed_at"),
            "department", "items", "unit_price_cents"
        )
        rows = []
        for row in formatted.collect():
            rows.append((row.order_id, row.placed_at, row.department,
                         int(row.items), int(row.unit_price_cents)))

        # Both inserts happen in the same transaction.
        with con:
            con.executemany(
                "INSERT OR IGNORE INTO orders VALUES (?, ?, ?, ?, ?)", rows
            )
            con.execute(
                "INSERT INTO processed_batches VALUES (?, ?)",
                (int(batch_id), len(rows))
            )

    # Stop once after saving a batch, to test recovery.
    if rows and failure_flag.exists() and not stopped_flag.exists():
        stopped_flag.touch()
        raise RuntimeError("Planned failure after saving the batch")


def start_stream():
    query = (unique_orders.writeStream
             .outputMode("append")
             .foreachBatch(save_batch)
             .option("checkpointLocation", str(checkpoint))
             .trigger(availableNow=True)
             .start())
    try:
        if not query.awaitTermination(120):
            raise TimeoutError("Streaming query took too long")
        return str(query.id)
    finally:
        if query.isActive:
            query.stop()

# Process the first delivery.
first_query_id = start_stream()
first_result = show_sql("SELECT * FROM orders ORDER BY order_id")
assert len(first_result) == 6
print("Orders saved after first delivery:", len(first_result))
display(first_result)


## 5. Test a failure during streaming

Now we send six more orders. The notebook deliberately raises an error after saving them in SQLite. This lets us check whether a restart can recover safely.


In [ ]:
# Second delivery has six new orders, one duplicate and one incomplete order.
send_orders(2, orders[6:12] + [orders[4]], json.dumps({"order_id": "MISSING"}))
failure_flag.touch()

spark.sparkContext.setLogLevel("OFF")
try:
    start_stream()
except StreamingQueryException as error:
    if "Planned failure after saving the batch" not in str(error):
        raise
    print("The planned streaming failure happened.")
else:
    raise AssertionError("The planned failure did not happen")
finally:
    spark.sparkContext.setLogLevel("ERROR")

assert stopped_flag.exists()
assert len(show_sql("SELECT * FROM orders")) == 12
print("Orders already saved:", 12)


## 6. Restart using the checkpoint

We send the remaining orders and restart Spark with the same checkpoint. Then we compare the saved data with the original orders to make sure no rows are missing or duplicated.


In [ ]:
# Send the last six orders with a duplicate and an invalid JSON line.
send_orders(3, orders[12:] + [orders[10]], "{bad json")
second_query_id = start_stream()
print("Same streaming query after restart:", first_query_id == second_query_id)
assert first_query_id == second_query_id

saved_orders = show_sql("SELECT * FROM orders ORDER BY order_id")
expected = pd.DataFrame(orders).sort_values("order_id").reset_index(drop=True)
pd.testing.assert_frame_equal(saved_orders, expected, check_dtype=False)
assert saved_orders.order_id.nunique() == 18

# Calculate the number of orders, items and sales for each department.
sales = show_sql("""
    SELECT department,
           COUNT(*) AS total_orders,
           SUM(items) AS total_items,
           SUM(items * unit_price_cents) AS total_paise
    FROM orders
    GROUP BY department
    ORDER BY department
""")
expected_total = sum(order["items"] * order["unit_price_cents"] for order in orders)
assert int(sales.total_paise.sum()) == expected_total
sales["total_rupees"] = sales["total_paise"] / 100
print("Orders recovered:", len(saved_orders))
display(sales)


In [ ]:
## 7. Final verification

# Running the stream again without new files should not change our results.
before = show_sql("SELECT * FROM orders ORDER BY order_id")
start_stream()
after = show_sql("SELECT * FROM orders ORDER BY order_id")
pd.testing.assert_frame_equal(before, after)

summary = pd.DataFrame({
    "Check": ["Original orders", "Recovered orders", "Duplicate IDs", "Total sales (paise)"],
    "Result": [len(orders), len(after), after.order_id.duplicated().sum(), int(sales.total_paise.sum())]
})
display(summary)
print("Verification successful: no orders were lost or repeated.")

saved_orders.to_csv(folder / "recovered_orders.csv", index=False)
sales.to_csv(folder / "department_sales.csv", index=False)
spark.stop()


## Result

We processed 18 sample shop orders using Spark Structured Streaming. Invalid records were rejected, duplicates were removed, and the correct orders were saved in SQLite. After an intentional failure, the stream restarted from its checkpoint and recovered all 18 orders without adding duplicates.


Reference: [Apache Spark Structured Streaming documentation](https://spark.apache.org/docs/4.0.1/streaming/apis-on-dataframes-and-datasets.html).
